# EDA Financial Discussions - Interactive Exploration

This notebook provides an interactive interface for exploring stock-related social media
discussion datasets. It demonstrates how to use the analysis functions from the `src/` modules
to perform dataset quality analysis, surge detection, and visualization.

**Sections:**
1. Dataset Loading
2. Structure Analysis
3. Missing Values
4. Engagement Distributions
5. Sentiment Analysis
6. Surge Threshold Exploration

---

## Setup

Import required libraries and analysis functions from the `src/` modules.

In [ ]:
import sys
import os

# Add project root to path so we can import from src/
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import analysis functions from src/ modules
from src.dataset_quality import (
    analyze_structure,
    compute_missing_values,
    analyze_time_coverage,
    compute_engagement_distributions,
    analyze_sentiment,
    assess_sentiment_reliability,
)
from src.surge_analysis import (
    normalize_engagement,
    compute_surge_labels,
    evaluate_surge_definitions,
    check_timestamp_resolution,
)
from src.visualization import (
    generate_engagement_distributions,
    generate_sentiment_distributions,
    generate_surge_frequency,
    generate_dataset_comparison,
)
from src.models import SurgeConfig

# Configure plotting
%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Dataset Loading

Load a candidate dataset for exploration. Replace the file path below with your
actual dataset (CSV, Parquet, etc.). The dataset should contain columns for:
- Stock ticker symbols
- Timestamps/dates
- Engagement metrics (likes, comments, retweets, upvotes)
- Text content (for sentiment analysis)

In [ ]:
# Load your dataset here
# Example: df = pd.read_csv('../data/stock_discussions.csv')

# For demonstration, create a sample dataset
np.random.seed(42)
n_rows = 1000

df = pd.DataFrame({
    'ticker': np.random.choice(['AAPL', 'TSLA', 'GME', 'AMZN', 'MSFT'], n_rows),
    'timestamp': pd.date_range('2023-01-01', periods=n_rows, freq='h'),
    'likes': np.random.exponential(scale=50, size=n_rows).astype(int),
    'comments': np.random.exponential(scale=10, size=n_rows).astype(int),
    'retweets': np.random.exponential(scale=20, size=n_rows).astype(int),
    'text': np.random.choice([
        'AAPL is going to the moon! Very bullish on this stock.',
        'Bearish outlook for TSLA, selling my position.',
        'GME short squeeze incoming, hold the line!',
        'AMZN earnings beat expectations, great quarter.',
        'Market is looking uncertain today.',
        'Just bought more shares, feeling confident.',
        'This stock is overvalued, time to sell.',
        'Neutral on this one, waiting for more data.',
    ], n_rows),
    'sentiment_score': np.random.normal(0.1, 0.4, n_rows),
})

# Introduce some missing values for realism
df.loc[np.random.choice(n_rows, 50, replace=False), 'comments'] = np.nan
df.loc[np.random.choice(n_rows, 30, replace=False), 'text'] = np.nan

print(f"Dataset loaded: {len(df)} rows, {len(df.columns)} columns")
df.head()

## 2. Structure Analysis

Analyze the dataset structure to understand schema, column types, record count,
and the number of unique stock tickers. This is the first step in assessing
whether a dataset is suitable for surge prediction.

In [ ]:
# Analyze dataset structure
structure = analyze_structure(df, ticker_col='ticker')

print(f"Record count: {structure['record_count']}")
print(f"Column count: {structure['column_count']}")
print(f"Unique tickers: {structure['ticker_count']}")
print(f"\nSchema:")
for col, dtype in structure['schema'].items():
    print(f"  {col}: {dtype}")

## 3. Missing Values

Compute per-column missing value percentages and identify high-risk columns
(those with >30% missing data). High-risk columns may require imputation
or exclusion from the analysis.

In [ ]:
# Compute missing values
missing_result = compute_missing_values(df)

print("Missing Value Percentages:")
print("-" * 40)
for col, pct in missing_result['missing_percentages'].items():
    flag = " ⚠️ HIGH RISK" if col in missing_result['high_risk_columns'] else ""
    print(f"  {col}: {pct:.2f}%{flag}")

if missing_result['high_risk_columns']:
    print(f"\n⚠️ High-risk columns (>30% missing): {missing_result['high_risk_columns']}")
else:
    print("\n✓ No high-risk columns detected.")

In [ ]:
# Visualize missing values
missing_pcts = missing_result['missing_percentages']
cols_with_missing = {k: v for k, v in missing_pcts.items() if v > 0}

if cols_with_missing:
    plt.figure(figsize=(8, 4))
    plt.barh(list(cols_with_missing.keys()), list(cols_with_missing.values()))
    plt.axvline(x=30, color='red', linestyle='--', label='High-risk threshold (30%)')
    plt.xlabel('Missing %')
    plt.title('Missing Values by Column')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found.")

## 4. Engagement Distributions

Compute and visualize summary statistics for engagement metrics. Understanding
the distribution of likes, comments, and retweets helps determine appropriate
percentile thresholds for surge detection.

In [ ]:
# Define engagement metric columns
engagement_cols = ['likes', 'comments', 'retweets']

# Compute engagement distributions
engagement_stats = compute_engagement_distributions(df, engagement_cols)

print("Engagement Distribution Statistics:")
print("=" * 60)
for metric, stats in engagement_stats.items():
    print(f"\n{metric}:")
    print(f"  Mean:   {stats['mean']:.2f}")
    print(f"  Median: {stats['median']:.2f}")
    print(f"  P90:    {stats['p90']:.2f}")
    print(f"  P95:    {stats['p95']:.2f}")
    print(f"  P99:    {stats['p99']:.2f}")

In [ ]:
# Visualize engagement distributions
fig, axes = plt.subplots(1, len(engagement_cols), figsize=(14, 4))

for i, col in enumerate(engagement_cols):
    if col in df.columns:
        ax = axes[i] if len(engagement_cols) > 1 else axes
        sns.histplot(df[col].dropna(), bins=50, ax=ax, kde=True)
        ax.set_title(f'{col.capitalize()} Distribution')
        ax.set_xlabel(col.capitalize())
        ax.set_ylabel('Count')
        # Mark percentile thresholds
        if col in engagement_stats:
            ax.axvline(engagement_stats[col]['p95'], color='orange',
                      linestyle='--', label='P95')
            ax.axvline(engagement_stats[col]['p99'], color='red',
                      linestyle='--', label='P99')
            ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Generate engagement distribution charts using the visualization module
# These are saved as PNG files in the output directory
output_dir = '../output'
os.makedirs(output_dir, exist_ok=True)

chart_paths = generate_engagement_distributions(engagement_stats, output_dir)
print(f"Generated {len(chart_paths)} engagement chart(s):")
for path in chart_paths:
    print(f"  {path}")

## 5. Sentiment Analysis

Analyze the sentiment distribution of text content in the dataset. This section
computes polarity scores, bullish/bearish ratios, and assesses sentiment
extraction reliability by comparing multiple methods.

In [ ]:
# Perform sentiment analysis on text content
sentiment_results = analyze_sentiment(df, text_col='text')

print("Sentiment Analysis Results:")
print("=" * 40)
print(f"Total texts analyzed: {sentiment_results['total_analyzed']}")
print(f"\nPolarity Scores:")
print(f"  Mean:   {sentiment_results['polarity_scores']['mean']:.4f}")
print(f"  Median: {sentiment_results['polarity_scores']['median']:.4f}")
print(f"  Std:    {sentiment_results['polarity_scores']['std']:.4f}")
print(f"\nSentiment Breakdown:")
print(f"  Bullish:  {sentiment_results['bullish_count']}")
print(f"  Bearish:  {sentiment_results['bearish_count']}")
print(f"  Neutral:  {sentiment_results['neutral_count']}")
print(f"  Bullish/Bearish Ratio: {sentiment_results['bullish_bearish_ratio']:.2f}")

In [ ]:
# Assess sentiment reliability by comparing VADER and TextBlob
reliability = assess_sentiment_reliability(df, text_col='text')

print("Sentiment Reliability Assessment:")
print("=" * 40)
if reliability['methods_compared']:
    print(f"Methods compared: {reliability['methods_compared']}")
    print(f"Agreement rate: {reliability['agreement_rate']:.2%}")
    print(f"Correlation: {reliability['correlation']:.4f}")
    print(f"VADER mean: {reliability['vader_mean']:.4f}")
    print(f"TextBlob mean: {reliability['textblob_mean']:.4f}")
    print(f"Texts compared: {reliability['total_compared']}")
else:
    print("Sentiment libraries not available. Install vaderSentiment and textblob.")

In [ ]:
# Generate sentiment distribution charts
sentiment_chart_paths = generate_sentiment_distributions(sentiment_results, output_dir)
print(f"Generated {len(sentiment_chart_paths)} sentiment chart(s):")
for path in sentiment_chart_paths:
    print(f"  {path}")

## 6. Surge Threshold Exploration

Explore different surge definitions by varying engagement percentile thresholds
and sentiment standard deviation thresholds. The goal is to find a definition
that produces a viable positive class (≥2% of records) while maintaining
meaningful surge detection.

A surge is defined as a post where:
- Engagement exceeds the configured percentile threshold (per-ticker normalized)
- Sentiment shift exceeds the configured standard deviation threshold
- Both conditions occur within a 24-hour time window

In [ ]:
# Check timestamp resolution for 24-hour window feasibility
ts_resolution = check_timestamp_resolution(df, timestamp_col='timestamp')

print("Timestamp Resolution Check:")
print("=" * 40)
print(f"Sufficient for 24h windows: {ts_resolution['sufficient']}")
print(f"Resolution: {ts_resolution['resolution']}")
print(f"Median gap (hours): {ts_resolution['median_gap_hours']:.2f}")
print(f"Min gap (hours): {ts_resolution['min_gap_hours']:.4f}")
print(f"Recommendation: {ts_resolution['recommendation']}")

In [ ]:
# Normalize engagement metrics per ticker
normalized_df = normalize_engagement(df, engagement_cols, ticker_col='ticker')

print("Per-ticker normalized engagement (first 5 rows):")
norm_cols = [f"{col}_normalized" for col in engagement_cols]
normalized_df[['ticker'] + engagement_cols + norm_cols].head()

In [ ]:
# Evaluate multiple surge definitions
percentiles = [0.90, 0.95, 0.99]
std_devs = [0.5, 1.0, 1.5]

surge_results = evaluate_surge_definitions(
    df,
    percentiles=percentiles,
    std_devs=std_devs,
    engagement_cols=engagement_cols,
    sentiment_col='sentiment_score',
    timestamp_col='timestamp',
    ticker_col='ticker',
)

print("Surge Definition Evaluation Results:")
print("=" * 70)
print(f"{'Percentile':<12} {'Std Devs':<10} {'Surge %':<10} {'Count':<8} {'Viable':<8} {'Imbalance'}")
print("-" * 70)
for result in surge_results:
    print(
        f"{result.config.engagement_percentile:<12.2f} "
        f"{result.config.sentiment_std_devs:<10.1f} "
        f"{result.surge_percentage:<10.2f} "
        f"{result.surge_count:<8} "
        f"{'✓' if result.is_viable else '✗':<8} "
        f"{result.class_imbalance_ratio:.1f}:1"
    )

In [ ]:
# Compute surge labels with a specific configuration
config = SurgeConfig(
    engagement_percentile=0.95,
    sentiment_std_devs=1.0,
    time_window_hours=24,
)

surge_labels = compute_surge_labels(
    df,
    config=config,
    engagement_cols=engagement_cols,
    sentiment_col='sentiment_score',
    timestamp_col='timestamp',
    ticker_col='ticker',
)

surge_count = surge_labels.sum()
total = len(surge_labels)
print(f"Surge labels (p=0.95, std=1.0):")
print(f"  Total posts: {total}")
print(f"  Surge posts: {surge_count}")
print(f"  Surge rate:  {surge_count / total * 100:.2f}%")

In [ ]:
# Visualize surge distribution across tickers
df_with_surge = df.copy()
df_with_surge['is_surge'] = surge_labels

surge_by_ticker = df_with_surge.groupby('ticker')['is_surge'].agg(['sum', 'count'])
surge_by_ticker['surge_pct'] = (surge_by_ticker['sum'] / surge_by_ticker['count']) * 100

plt.figure(figsize=(8, 4))
surge_by_ticker['surge_pct'].plot(kind='bar')
plt.axhline(y=2.0, color='red', linestyle='--', label='Viability threshold (2%)')
plt.title('Surge Rate by Ticker')
plt.xlabel('Ticker')
plt.ylabel('Surge %')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Generate surge frequency chart using the visualization module
surge_chart_path = generate_surge_frequency(df_with_surge, output_dir)
print(f"Generated surge frequency chart: {surge_chart_path}")

## Summary

This notebook demonstrated how to use the `src/` analysis modules interactively:

- **`src.dataset_quality`**: Structure analysis, missing values, engagement distributions, sentiment analysis
- **`src.surge_analysis`**: Per-ticker normalization, surge label computation, threshold evaluation
- **`src.visualization`**: Chart generation for engagement, sentiment, and surge frequency

For the full automated pipeline, run `python main.py` from the project root.
The pipeline executes all analysis steps in sequence and produces the complete
markdown report with charts.